# Cert Prep — 06: Performance & Optimization

**Exam weight: ~15%**

Topics covered:
- Caching: persist(), cache(), StorageLevel
- Broadcast joins: when and how
- AQE: coalescing, join conversion, skew handling
- Shuffling: causes, cost, avoidance strategies
- Partitioning: repartition vs coalesce
- explain() — reading query plans
- Common bottlenecks: data skew, small files, spill


In [ ]:
import os, time, pathlib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER = 'spark://spark-master:7077'

spark = (
    SparkSession.builder
    .appName('cert-prep')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.sql.warehouse.dir', '/workspace/data/training_warehouse')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

# Sample dataset used throughout
data = [
    (1, "Alice",  "Engineering", 95000.0, "2021-03-15", 4),
    (2, "Bob",    "Marketing",   72000.0, "2020-07-01", 3),
    (3, "Carol",  "Engineering", 105000.0,"2019-11-20", 5),
    (4, "Dave",   "Marketing",   68000.0, "2022-01-10", 2),
    (5, "Eve",    "Engineering", 88000.0, "2021-09-05", 4),
    (6, "Frank",  "HR",          61000.0, "2020-04-18", 3),
    (7, "Grace",  "HR",          None,    "2023-02-28", 1),
    (8, "Hank",   "Engineering", 112000.0,"2018-06-12", 5),
    (9, "Iris",   "Marketing",   None,    "2022-11-03", 2),
    (10,"Jack",   "HR",          59000.0, "2023-08-01", 1),
]
schema = StructType([
    StructField("id",         IntegerType(),  False),
    StructField("name",       StringType(),   True),
    StructField("dept",       StringType(),   True),
    StructField("salary",     DoubleType(),   True),
    StructField("hire_date",  StringType(),   True),
    StructField("perf_score", IntegerType(),  True),
])
df = spark.createDataFrame(data, schema)
df.cache().count()
print("Dataset ready")
df.show()

In [ ]:
# ── Caching / Persist ────────────────────────────────────────────────────────
print("=== Caching ===")
from pyspark import StorageLevel

# cache() = persist(MEMORY_AND_DISK)
df_cached = df.cache()
df_cached.count()  # materialize
print(f"Is cached: {df_cached.is_cached}")

# StorageLevels for different use cases:
print()
print("StorageLevel.MEMORY_ONLY        — fastest, fails if not enough RAM")
print("StorageLevel.MEMORY_AND_DISK    — spills to disk if RAM full (default cache)")
print("StorageLevel.DISK_ONLY          — for large DFs that don't fit in RAM")
print("StorageLevel.MEMORY_ONLY_SER    — compressed bytes in RAM, less space, more CPU")
print("StorageLevel.OFF_HEAP           — off-heap memory, reduces GC pressure")
print()

df_disk = df.persist(StorageLevel.DISK_ONLY)
df_disk.count()
print("Persisted to disk.")

# Always unpersist when done to free memory
df_cached.unpersist()
df_disk.unpersist()
print("Unpersisted.")

In [ ]:
# ── Broadcast Joins ──────────────────────────────────────────────────────────
print("=== Broadcast Joins ===")

dept_small = spark.createDataFrame([
    ("Engineering", "San Francisco"),
    ("Marketing",   "New York"),
    ("HR",          "Austin"),
], ["dept", "location"])

# Auto-broadcast: Spark broadcasts tables < spark.sql.autoBroadcastJoinThreshold (10MB)
# Manual broadcast hint:
from pyspark.sql.functions import broadcast

result = df.join(broadcast(dept_small), on="dept", how="left")
result.select("name","dept","location").show()

print("Plan — look for BroadcastHashJoin:")
result.explain()
print()
print("Use broadcast when:")
print("  - One side is small (< ~10 MB, configurable)")
print("  - spark.sql.autoBroadcastJoinThreshold = 10485760 (10 MB default)")
print("  - Eliminates shuffle on the large side → major speedup")
print()
print("Disable broadcast:")
print("  spark.conf.set('spark.sql.autoBroadcastJoinThreshold', -1)")

In [ ]:
# ── AQE — Adaptive Query Execution ───────────────────────────────────────────
print("=== AQE ===")
print(f"AQE enabled: {spark.conf.get('spark.sql.adaptive.enabled')}")
print()
print("AQE has 3 features:")
print()
print("1. Coalescing shuffle partitions")
print("   spark.sql.adaptive.coalescePartitions.enabled = true")
print("   Merges small post-shuffle partitions into larger ones")
print("   Prevents 200 tiny partitions for a 1 MB result")
print()
print("2. Converting sort-merge join to broadcast join")
print("   After shuffle, if one side is small enough → convert to BHJ")
print("   No pre-execution statistics needed")
print()
print("3. Skew join optimization")
print("   spark.sql.adaptive.skewJoin.enabled = true")
print("   Detects skewed partitions, splits them, processes in parallel")
print("   Threshold: spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes")
print()

# Demonstrate: run a groupBy and observe AQE coalescing
import time
t0 = time.time()
result = df.groupBy("dept","perf_score").agg(F.sum("salary")).collect()
print(f"GroupBy completed in {time.time()-t0:.2f}s")

In [ ]:
# ── Reading Query Plans ──────────────────────────────────────────────────────
print("=== Reading explain() Output ===")

q = (df
    .filter(F.col("salary") > 70000)
    .join(broadcast(
        spark.createDataFrame([("Engineering","critical"),("HR","support")],
                              ["dept","tier"])
    ), on="dept")
    .groupBy("dept","tier")
    .agg(F.avg("salary").alias("avg_salary")))

print("=== Simple (default) ===")
q.explain()

print("\n=== Formatted ===")
q.explain("formatted")

In [ ]:
# ── Data Skew ────────────────────────────────────────────────────────────────
print("=== Data Skew ===")
print()
print("Data skew: some partitions have far more data than others")
print("Symptom: most tasks finish in 2s, one task takes 200s")
print()
print("Solutions:")
print()
print("1. AQE skew handling (automatic, Spark 3+)")
print("   spark.sql.adaptive.skewJoin.enabled = true")
print()
print("2. Salting — add random key to distribute hot key")
print("   df.withColumn('salt', (F.rand() * N).cast('int'))")
print("   join on (original_key, salt) after salting both sides")
print()
print("3. Broadcast join — if small side, eliminate join shuffle entirely")
print()
print("4. Repartition by join key before join:")
print("   df.repartition(200, 'join_key').join(...)")
print()

# Detect skew: partition size distribution
rdd_sizes = df.repartition(4).rdd.mapPartitions(lambda it: [sum(1 for _ in it)]).collect()
print(f"Partition sizes: {rdd_sizes}")
print(f"Max/Min ratio: {max(rdd_sizes)/max(min(rdd_sizes),1):.1f}x")